# A/B/C/D 실제 결과 비교 · 원시 로그 검증

05의 완료 결과를 읽어 원문·프롬프트·점수·토큰·비용을 검증하고 그래프를 만든다. **API 호출과 모델 추론은 하지 않는다.**
기본값은 가장 최근에 완료한 일반 시스템 런 또는 단계 검증의 별도 평가다.
첫 셀의 `SYSTEM_RUN`에는 과거 결과 폴더명, `staged_*` 묶음, 개별 `dev/ABD_input_fix` 경로를 지정할 수 있다.

단계 검증 결과에서는 입력 검사·C 개별효과·D 정책 선택·질문별 분포·대응 비교·실제 API ledger도 확인한다.
분석 CSV와 그림은 새 `results/comparison_<UTC시간>_<ID>/`에 저장하며 원본 결과를 덮어쓰지 않는다.
서로 다른 모델의 토큰은 합산하지 않고, 온라인 API 비용과 로컬 GPU·준비 비용을 구분한다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import uuid

import nbformat
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager, ticker
from IPython.display import display, Markdown

ROOT = Path.cwd() if (Path.cwd() / "rag_baselines").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
# Refresh an ABC-only loader cached by an earlier notebook kernel.
import importlib
importlib.reload(importlib.import_module("rag_baselines.comparison"))
from rag_baselines.comparison import load_comparison, resolve_comparison_run
from rag_baselines.data import file_sha256, read_json, read_jsonl, write_json

SYSTEM_RUN = None  # None=가장 최근 완료 런. staged_* 폴더나 개별 런 경로도 가능.
RUN_DIR, STUDY_DIR = resolve_comparison_run(ROOT, SYSTEM_RUN)
SYSTEM_RUN = str(RUN_DIR.relative_to(ROOT/'results'))
RUN_MANIFEST = read_json(RUN_DIR / "manifest.json")
METHODS = RUN_MANIFEST.get("methods", ["A", "B", "C"])
print(f"분석 원본: {RUN_DIR.name} | {RUN_MANIFEST['mode']} | methods={METHODS}")
NAMES = {"A": "A · 기본 RAG", "B": "B · 재정렬 RAG", "C": "C · Jev 트리", "D": "D · Laya 로컬"}
COLORS = {"A": "#416D95", "B": "#BE7D32", "C": "#537D59", "D": "#8565A8"}
TASK_MARKERS = {"nq": ("D", -.15), "hotpotqa": ("o", .15)}

font_path = Path("C:/Windows/Fonts/malgun.ttf")
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams.update({
    "font.size": 11, "axes.unicode_minus": False, "axes.spines.top": False,
    "axes.spines.right": False, "axes.edgecolor": "#CDD5DC", "axes.labelcolor": "#364656",
    "text.color": "#1D2D3A", "xtick.color": "#364656", "ytick.color": "#364656",
    "figure.facecolor": "#FFFFFF", "axes.facecolor": "#FFFFFF", "savefig.facecolor": "#FFFFFF",
})


## 원문과 원시 결과를 다시 대조

`load_comparison`은 corpus·질문·라벨 해시, 원문 근거, 공동 프롬프트, A/B 동일 후보,
질문별 EM/F1·근거 회수, 요청별 usage에서 다시 계산한 API 비용과 시간 기록을 확인한다.
D의 로컬 usage와 잘림 보고도 요청 기록·결과 요약 사이에서 대조한다. 중복·누락 방법·잘못된 비용·바뀐 프롬프트가 있으면 그림 전에 중단한다.
로컬 모델 토큰은 처리량 설명용이며 Jev·GPT 토큰과 합산하지 않는다.


In [ ]:
manifest, rows, source_hashes = load_comparison(ROOT, RUN_DIR)
df = pd.DataFrame(rows).sort_values(["task", "question_id", "method", "repeat"]).reset_index(drop=True)
df["overhead_seconds"] = (
    df["online_seconds"] - df["retrieval_seconds"] - df["generation_seconds"]
).clip(lower=0)
question_count = df["question_id"].nunique()
repeats = sorted(df["repeat"].unique().tolist())
corpus_count = manifest["index"]["signature"]["count"]
metrics = [
    "answer_em", "answer_f1", "complete_evidence_hit", "best_group_coverage",
    "online_seconds", "retrieval_seconds", "generation_seconds", "overhead_seconds", "diagnostic_capture_seconds",
    "llm_input_tokens", "llm_output_tokens", "jev_input_tokens", "jev_requests", "laya_input_tokens",
    "laya_decision_calls", "laya_truncated_calls", "decision_model_calls", "evidence_count",
    "retrieval_api_cost_usd", "generation_api_cost_usd", "online_api_cost_usd",
]
summary = pd.DataFrame(index=pd.Index(METHODS, name="method"))
for method in METHODS:
    observations = df.loc[df.method == method]
    summary.loc[method, "questions"] = observations.question_id.nunique()
    summary.loc[method, "observations"] = len(observations)
    for metric in metrics:
        summary.loc[method, metric] = observations[metric].mean()
        summary.loc[method, f"{metric}_n"] = observations[metric].notna().sum()

print(f"검증 통과: {SYSTEM_RUN} | {manifest['split']} {question_count}문항 | "
      f"방법당 {len(repeats)}회 | 공통 {corpus_count:,}문단 | methods={METHODS}")
display(pd.DataFrame([{"source": path, "sha256": sha} for path, sha in source_hashes.items()]))
display(df[["task", "question_id", "question"]].drop_duplicates().head(8).reset_index(drop=True))
print("질문 표는 최대 8개 미리보기다. 전체 질문별 결과와 로그 연결은 CSV/JSONL에 보존된다.")
raw_session = manifest.get('raw_log')
RAW_CAPTURE_ENABLED = bool(raw_session.get('enabled')) if isinstance(raw_session, dict) else bool(raw_session)
print('추가 원시 기록:', '포함 (관측 시간에 기록 비용 포함)' if RAW_CAPTURE_ENABLED else '없음 또는 과거 런')
if RAW_CAPTURE_ENABLED:
    print('Raw session:',raw_session['session'] if isinstance(raw_session, dict) else raw_session)
    display(df[['task','question_id','method','repeat','diagnostic_capture_seconds','raw_events_path']].head(12))
if 'D' in METHODS:
    print('D model runtime:',manifest['d']['runtime'])
    print('C/D preparation: Jev-prepared tree is shared; it is counted once.')


## 요약과 질문별 관측

각 수치는 같은 질문에 대한 균형 잡힌 반복의 산술 평균이다. `*_n`은 실제 관측 수다.
답변 EM/F1과 정답 근거 회수는 별개다. NQ 정답 span을 찾았더라도 답에 필요한 사실이 부족하면 생성은 실패할 수 있다.

온라인 비용은 외부 API usage × 해당 런의 가격표다. C는 Jev 탐색 API와 GPT 생성, D는 GPT 생성만 online API 비용에 들어간다. D 로컬 GPU 시간·메모리와 전기 비용은 별도로 보고하고 API 금액에 환산하지 않는다. C와 D는 공통 Jev 준비 트리를 공유하므로 준비 비용은 한 번만 기록한다.

C Jev input tokens, D Laya input tokens, GPT input tokens는 각각 다른 tokenizer/provider 기준이다. 서로 더하거나 직접 토큰 단가가 같다고 해석하지 않는다.


In [ ]:
summary_columns = ["questions", "observations", "answer_em", "answer_f1", "complete_evidence_hit",
                   "online_seconds", "retrieval_seconds", "llm_input_tokens", "jev_input_tokens", "laya_input_tokens",
                   "jev_requests", "laya_decision_calls", "evidence_count", "online_api_cost_usd"]
display(summary[summary_columns].style.format({
    "questions": "{:.0f}", "observations": "{:.0f}", "answer_em": "{:.0%}",
    "answer_f1": "{:.3f}", "complete_evidence_hit": "{:.0%}",
    "online_seconds": "{:.3f}", "retrieval_seconds": "{:.3f}", "llm_input_tokens": "{:,.1f}",
    "jev_input_tokens": "{:,.1f}", "laya_input_tokens": "{:,.1f}", "jev_requests": "{:.1f}",
    "laya_decision_calls": "{:.1f}", "evidence_count": "{:.1f}",
    "online_api_cost_usd": "$ {:,.7f}",
}, na_rep="미측정 / 해당 없음"))

detail_columns = ["task", "method", "answer", "answer_em", "answer_f1", "complete_evidence_hit",
                  "evidence_count", "online_seconds", "llm_input_tokens", "jev_input_tokens",
                  "laya_input_tokens", "jev_requests", "laya_decision_calls", "laya_truncated_calls",
                  "online_api_cost_usd", "decision_stop_reason"]
display(df[detail_columns].head(12).style.format({
    "answer_em": "{:.0f}", "answer_f1": "{:.3f}", "complete_evidence_hit": "{:.0f}",
    "online_seconds": "{:.3f}", "llm_input_tokens": "{:,.0f}", "jev_input_tokens": "{:,.0f}",
    "laya_input_tokens": "{:,.0f}", "jev_requests": "{:.0f}", "laya_decision_calls": "{:.0f}",
    "laya_truncated_calls": "{:.0f}", "online_api_cost_usd": "$ {:,.7f}",
}, na_rep="—"))

print("질문별 표는 최대 12개 결과 미리보기다. 전체 결과는 comparison.csv에 저장한다.")
task_summary = df.groupby(["task", "method"]).agg(
    questions=("question_id", "nunique"), observations=("question_id", "size"),
    correct_answers=("answer_em", "sum"), answer_em=("answer_em", "mean"), answer_f1=("answer_f1", "mean"),
    complete_evidence_hit=("complete_evidence_hit", "mean"), online_seconds=("online_seconds", "mean"),
    online_api_cost_usd=("online_api_cost_usd", "mean"))
display(task_summary.style.format({"answer_em":"{:.1%}", "answer_f1":"{:.3f}",
    "complete_evidence_hit":"{:.1%}", "online_seconds":"{:.3f}", "online_api_cost_usd":"$ {:.7f}"},na_rep="미측정"))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
EXPORT_DIR = ROOT / "results" / f"comparison_{stamp}_{uuid.uuid4().hex[:8]}"
EXPORT_DIR.mkdir(parents=True)
df.to_csv(EXPORT_DIR / "comparison.csv", index=False, encoding="utf-8-sig")
summary.to_csv(EXPORT_DIR / "summary.csv", encoding="utf-8-sig")
task_summary.to_csv(EXPORT_DIR / "task_summary.csv", encoding="utf-8-sig")


In [ ]:
def panel(ax, metric, title, unit, formatter, ceiling=None, not_applicable=()):
    values = summary[metric].reindex(METHODS).to_numpy(dtype=float)
    numeric = values[np.isfinite(values)]
    observed_max = df.groupby(["method", "task"])[metric].mean().max()
    upper = ceiling or max(float(observed_max) if pd.notna(observed_max) else 0,
                           float(numeric.max()) if len(numeric) else 0, 1e-9) * 1.32
    positions = np.arange(len(METHODS))
    ax.bar(positions, values, color=[COLORS[m] for m in METHODS], width=.58, alpha=.86)
    for x, method in enumerate(METHODS):
        task_means = []
        for task, (marker, offset) in TASK_MARKERS.items():
            raw = df.loc[(df.method == method) & (df.task == task), metric].dropna()
            if not raw.empty:
                task_means.append(raw.mean())
                ax.scatter(x + offset, raw.mean(), marker=marker, s=38,
                           facecolor="white", edgecolor=COLORS[method], linewidth=1.4, zorder=3, clip_on=False)
        if np.isfinite(values[x]):
            label = formatter(values[x])
            if metric == "answer_em":
                raw = df.loc[df.method == method, metric].dropna()
                label += f"\n({raw.sum():g}/{len(raw)})"
            label_base = max([values[x], *task_means])
            ax.text(x, label_base + upper * .035, label,
                    ha="center", va="bottom", fontsize=11, fontweight="bold")
        else:
            label = "해당 없음" if method in not_applicable else "미측정"
            ax.text(x, upper * .055, label, ha="center", fontsize=9)
    ax.set_xticks(positions, [NAMES[m] for m in METHODS])
    ax.set_ylim(0, upper)
    ax.set_ylabel(unit)
    ax.set_title(title, loc="left", pad=14, fontsize=12, fontweight="bold")
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#E5EAF0", linewidth=.8)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(5))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.subplots_adjust(left=.075, right=.97, top=.82, bottom=.15, hspace=.62, wspace=.3)
fig.suptitle(f"A/B/C/D {manifest['split']} 실측 · 품질, 시간, 토큰, 비용", x=.07, y=.965,
             ha="left", fontsize=20, fontweight="bold")
fig.text(.07, .914, f"공통 {corpus_count:,}문단 · {question_count}문항 × 방법당 {len(repeats)}회 · "
         f"생성기 {manifest['generation']['model']}", fontsize=11, color="#506273")
fig.text(.07, .878, "막대 = 전체 평균    ◆ NQ 평균    ● HotpotQA 평균    |    A/B 로컬 dense 검색 · C 원격 Jev · D 로컬 Laya",
         fontsize=10, color="#506273")
panel(axes[0, 0], "answer_em", "01  답변 정확도 · Exact Match", "정답 비율",
      lambda v: f"{v:.1%}", ceiling=1.16)
axes[0, 0].yaxis.set_major_formatter(ticker.PercentFormatter(1))
axes[0, 0].set_yticks([0, .25, .5, .75, 1])
panel(axes[0, 1], "online_seconds", "02  실제 전체 응답 시간" + (" · 원시 기록 포함" if RAW_CAPTURE_ENABLED else ""), "초 / 질문",
      lambda v: f"{v:.2f}초")
panel(axes[1, 0], "llm_input_tokens", "03  최종 LLM 입력", "GPT API 입력 토큰 / 질문",
      lambda v: f"{v:,.1f}")
panel(axes[1, 1], "online_api_cost_usd", "04  외부 온라인 API 비용", "USD / 질문",
      lambda v: f"$ {v:.7f}")
axes[1, 1].yaxis.set_major_formatter(ticker.StrMethodFormatter("${x:.4f}"))
fig.text(.07, .088, "EM은 정규화 정답 완전일치이며 (정답 수/응답 수)를 표시한다. 선택 문항의 관측 결과다.",
         fontsize=10, color="#506273")
fig.text(.07, .054, "비용은 C Jev 탐색 + GPT 생성 API usage다. D는 로컬 API 요금 0이며 GPU·준비·사람 비용은 포함하지 않는다.",
         fontsize=10, color="#506273")
fig.savefig(EXPORT_DIR / "overview.png", dpi=160)
fig.savefig(EXPORT_DIR / "overview.svg")
plt.show()


## 탐색 비용과 반환 근거를 함께 확인

단계 시간은 같은 시스템 런의 검색·생성·기타 overhead로 나눈다. 품질 캐시와 별도 지연 런을 섞지 않는다.
C의 Jev API 입력과 D의 로컬 Laya 입력은 각 모델이 실제 받은 입력 usage다. tokenizer가 다르므로 두 값을 한 토큰 총합으로 더하지 않는다. D의 판단 API 호출은 로컬 inference call 수로 구분한다.

Jev 트리 준비는 C와 D가 공유한다. 최종 GPT 입력이 줄어도 사전 준비·로컬 추론·GPU 비용이 사라지는 것은 아니다.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 13))
fig.subplots_adjust(left=.08, right=.97, top=.91, bottom=.1, hspace=.62, wspace=.3)
fig.suptitle("검색 방식별 부담 · 최종 LLM 입력만으로 판단하지 않기", x=.08, y=.975,
             ha="left", fontsize=19, fontweight="bold")
fig.text(.08, .943, f"동일 시스템 런 {SYSTEM_RUN} · C/D는 같은 준비 트리", color="#506273")
x = np.arange(len(METHODS))

ax = axes[0, 0]
retrieval = summary["retrieval_seconds"].to_numpy(float)
generation = summary["generation_seconds"].to_numpy(float)
overhead = summary["overhead_seconds"].to_numpy(float)
ax.bar(x, retrieval, color=[COLORS[m] for m in METHODS], width=.58, label="검색")
ax.bar(x, generation, bottom=retrieval, color="#BCC9D6", width=.58, label="프롬프트 + GPT API")
ax.bar(x, overhead, bottom=retrieval + generation, color="#E0E6EB", width=.58, label="기타 overhead")
ax.set_title("01  전체 응답 시간의 구성", loc="left", pad=14, fontsize=12, fontweight="bold")
ax.set_ylabel("초 / 질문")
ax.set_ylim(0, summary.online_seconds.max() * 1.3)
ax.legend(frameon=False, fontsize=9, loc="upper left")
for i, value in enumerate(summary.online_seconds.reindex(METHODS)):
    ax.text(i, value + ax.get_ylim()[1] * .035, f"{value:.2f}초", ha="center", fontweight="bold")

ax = axes[0, 1]
llm_cost = summary["generation_api_cost_usd"].reindex(METHODS).to_numpy(float)
decision_cost = summary["retrieval_api_cost_usd"].reindex(METHODS).to_numpy(float)
ax.bar(x, llm_cost, color="#BCC9D6", width=.58, label="GPT 생성 API")
ax.bar(x, decision_cost, bottom=llm_cost, color=COLORS["C"], width=.58, label="Jev 탐색 API")
ax.set_title("02  외부 API 비용의 구성", loc="left", pad=14, fontsize=12, fontweight="bold")
ax.set_ylabel("USD / 질문")
ax.set_ylim(0, summary.online_api_cost_usd.max() * 1.35)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("${x:.4f}"))
ax.legend(frameon=False, fontsize=9, loc="upper left")
for i, value in enumerate(summary.online_api_cost_usd.reindex(METHODS)):
    if pd.notna(value):
        ax.text(i, value + ax.get_ylim()[1] * .035, f"$ {value:.7f}", ha="center", fontweight="bold")

panel(axes[1, 0], "jev_input_tokens", "03  C · Jev 원격 입력", "Jev 제공사 입력 tokens / 질문", lambda v: f"{v:,.1f}", not_applicable=("A", "B", "D"))
axes[1, 0].yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))

if "D" in METHODS:
    panel(axes[1, 1], "laya_input_tokens", "04  D · Laya 로컬 입력", "Laya tokenizer input tokens / 질문", lambda v: f"{v:,.1f}", not_applicable=("A", "B", "C"))
    axes[1, 1].yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
else:
    axes[1, 1].axis("off")
    axes[1, 1].text(.03, .92, "04  D · Laya 로컬 입력", transform=axes[1, 1].transAxes, fontweight="bold")
    axes[1, 1].text(.03, .75, "이전 A/B/C 런에는 D가 포함되지 않았다.", transform=axes[1, 1].transAxes)

panel(axes[2, 0], "evidence_count", "05  실제 LLM에 전달한 원문", "원문 문단 수 / 질문", lambda v: f"{v:.1f}개")
panel(axes[2, 1], "decision_model_calls", "06  트리 판단 모델 호출", "C Jev 요청 / D Laya local call per question", lambda v: f"{v:.1f}", not_applicable=("A", "B"))

for ax in axes.flat:
    if ax.axison:
        ax.set_xticks(x, [NAMES[m] for m in METHODS])
        ax.set_axisbelow(True)
        ax.grid(axis="y", color="#E5EAF0", linewidth=.8)
        ax.yaxis.set_major_locator(ticker.MaxNLocator(5))
fig.text(.08, .046, "C Jev와 D Laya의 입력 token 수는 각각의 tokenizer 기준이며 직접 합산·단가 비교하지 않는다. 로컬 GPU 비용은 미측정.",
         fontsize=10, color="#506273")
fig.savefig(EXPORT_DIR / "retrieval_detail.png", dpi=160)
fig.savefig(EXPORT_DIR / "retrieval_detail.svg")
plt.show()


## 사전 준비 비용

C와 D는 같은 Jev로 만든 하나의 트리를 공유한다. 이 비교는 온라인 판단 모델 차이를 다루므로, 공유 트리 준비 비용은 한 번만 표시한다. D만으로 새 트리를 자동 분류하는 준비 방식의 비용/품질을 검증한 것은 아니다. 로컬 Laya model load/warmup은 retrieval 시간에서 제외되어 별도 기록된다.
A/B는 dense index를 공유한다. 사람의 트리 작성·유지보수와 로컬 GPU 금전 비용은 미측정이다.


In [ ]:
prep = manifest["c"]["preparation"]
preparation_rows = [
    {"component": "A/B 공통 dense index", "original_prepare_seconds": manifest["index"]["build_seconds"],
     "api_cost_usd": None, "API_input_tokens": None,
     "local_embedding_nonpad_tokens": manifest["index"]["document_tokens"]["nonpad"],
     "note": "로컬 모델; GPU 금전 비용 미측정"},
    {"component": "C/D shared Jev 자료 트리", "original_prepare_seconds": prep["prepare_seconds"],
     "api_cost_usd": prep["api_cost_usd"], "API_input_tokens": prep["api_usage"]["input_tokens"],
     "local_embedding_nonpad_tokens": None, "note": "C와 D 공통 1회 준비 비용; 사람의 taxonomy 작성 시간 미측정"},
]
if "D" in METHODS:
    runtime = manifest["d"]["runtime"]
    preparation_rows.append({"component": "D local Laya model load + warmup", 
        "original_prepare_seconds": runtime["load_seconds"] + runtime["warmup_seconds"],
        "api_cost_usd": 0.0, "API_input_tokens": None, "local_embedding_nonpad_tokens": None,
        "note": f"online retrieval에서 제외; {runtime['actual_device']} / {runtime['dtype']}"})
preparation = pd.DataFrame(preparation_rows)
display(preparation.style.format({"original_prepare_seconds": "{:.3f}",
    "api_cost_usd": "$ {:.7f}", "API_input_tokens": "{:,.0f}", "local_embedding_nonpad_tokens": "{:,.0f}"},
    na_rep="미측정 / 해당 없음"))
preparation.to_csv(EXPORT_DIR / "preparation.csv", index=False, encoding="utf-8-sig")

## 단계 검증의 입력 검사와 개발 선택

이 절은 단계 검증 묶음의 저장 결과가 있을 때 실행한다. 현재 설정 파일 대신 **해당 실행에 기록된 설정**을 사용한다.
입력 재생은 추론·지연 결과와 구분하고, 소수의 판단 검사는 일반 정확도로 해석하지 않는다.

In [ ]:
from rag_baselines.validation_study import summary_rows, aggregate
STUDY = manifest.get('study') if STUDY_DIR is not None else None
dev_summary = None
comparison_frames = []
def show_summary(table):
    table = table.copy()
    table['label'] = table['arm'] if table['method'].nunique()==1 else table['method']
    metrics = ['questions','answer_em','answer_f1','complete_evidence_hit','complete_candidate_hit',
               'online_seconds','retrieval_seconds','llm_input_tokens','decision_input_tokens',
               'decision_calls','evidence_count','online_api_cost_usd']
    display(table.set_index('label')[metrics].T.round(6))

In [ ]:
if STUDY_DIR is not None:
    if (STUDY_DIR/'d_input_replay_summary.json').exists():
        replay = read_json(STUDY_DIR/'d_input_replay_summary.json')
        assert replay['accepted'] == replay['decisions'] and not replay['failures']
        display(pd.DataFrame([{k:v for k,v in replay.items() if k not in ('failures','source_sha256')}]))
    if (STUDY_DIR/'decision_validation.csv').exists():
        diagnostics = pd.read_csv(STUDY_DIR/'decision_validation.csv')
        display(diagnostics.groupby('stage').agg(cases=('correct','size'), accuracy_at_original_cutoff=('correct','mean')))
    if (STUDY_DIR/'score_order_summary.json').exists():
        display(read_json(STUDY_DIR/'score_order_summary.json'))
    if (STUDY_DIR/'dev_c_summary.csv').exists():
        dev_summary = pd.read_csv(STUDY_DIR/'dev_c_summary.csv')
        frozen = read_json(STUDY_DIR/'frozen_c_policy.json')
        assert set(dev_summary.arm) == set(STUDY['variants'])
        show_summary(dev_summary)
        print('개발에서 고정한 C:', frozen['selected_arm'])
    for split, name in [('dev','ABD_input_fix'),('dev','D_candidate'),('eval','ABCD_frozen')]:
        directory = STUDY_DIR/split/name
        if (directory/'completed.json').exists():
            frame = pd.read_csv(directory/'questions.csv')
            frame['split'] = split
            comparison_frames.append(frame)
            show_summary(aggregate(frame))
    if (STUDY_DIR/'frozen_d_policy.json').exists():
        show_summary(pd.read_csv(STUDY_DIR/'dev_d_summary.csv'))
        print('개발에서 고정한 D:', read_json(STUDY_DIR/'frozen_d_policy.json')['selected_arm'])
else:
    print('일반 시스템 런: 별도 단계 검증 결과 없음.')

## 4. 저장 자료 대조와 비교 시각화

결과 JSONL에서 지표를 다시 계산한다. 비용은 기록된 단가를 사용한 API 사용량 추정이며 GPU·전력·인건비를 제외한다.
막대는 질문 평균, 시간은 실제 검색 및 생성의 전체 관측 시간이다. 반복 1회의 작은 실험이므로 우위 확정에 사용하지 않는다.

In [ ]:
def run_answer_audit(directory):
    manifest = read_json(directory/'manifest.json')
    if 'answer_audit' in manifest: return manifest['answer_audit']
    sources = list((ROOT/manifest['raw_log']/'sources').glob('*_answer_audit.json'))
    if len(sources)!=1: raise ValueError('Need the exact answer audit snapshot for this run.')
    return read_json(sources[0])
recomputed = []
for arm in (STUDY['variants'] if dev_summary is not None else []):
    directory = STUDY_DIR/'dev'/arm
    completed = read_json(directory/'completed.json')
    assert completed['system_sha256'] == file_sha256(directory/'system.jsonl')
    records = read_jsonl(directory/'system.jsonl')
    assert len(records) == completed['records']
    keys = [(r['retrieval']['question_id'],r['retrieval']['repeat']) for r in records]
    assert len(keys) == len(set(keys))
    recomputed.extend(summary_rows(records, arm, run_answer_audit(directory)))
if recomputed:
    recomputed_frame = pd.DataFrame(recomputed)
    fresh = aggregate(recomputed_frame)
    pd.testing.assert_frame_equal(fresh.reset_index(drop=True), dev_summary.reset_index(drop=True), check_dtype=False, atol=1e-10)
    display(recomputed_frame[recomputed_frame['gold_label_needs_semantic_review']][['arm','question_id','answer','answer_em','complete_evidence_hit']])
    print('All four development arms reconciled from their original JSONL records.')
for frame in comparison_frames:
    directory = STUDY_DIR/frame['split'].iloc[0]/frame['arm'].iloc[0]
    completed = read_json(directory/'completed.json')
    assert completed['system_sha256'] == file_sha256(directory/'system.jsonl')
    records = read_jsonl(directory/'system.jsonl')
    assert len(records) == completed['records']
    keys = [(r['retrieval']['question_id'],r['retrieval']['method'],r['retrieval']['repeat']) for r in records]
    assert len(keys) == len(set(keys))
    restored = pd.DataFrame(summary_rows(records, frame['arm'].iloc[0], run_answer_audit(directory)))
    keys = ['task','question_id','method','repeat']
    pd.testing.assert_frame_equal(
        restored.sort_values(keys).reset_index(drop=True),
        frame.drop(columns='split').sort_values(keys).reset_index(drop=True),
        check_dtype=False, atol=1e-10)
    pd.testing.assert_frame_equal(aggregate(restored), pd.read_csv(directory/'summary.csv'), check_dtype=False, atol=1e-10)
    if {'A','B'} <= set(frame.method):
        _, reconciled, _ = load_comparison(ROOT,directory)
        assert len(reconciled)==len(records)
    print(directory.name, ': results and stored summary match.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
colors = list(COLORS.values())
def comparison_plot(table, title, destination):
    table = table.copy()
    display_names={'A':'A · 기본 RAG','B':'B · 재정렬 RAG','C':'C · Jev 트리','D':'D · Laya 트리'}
    table['label'] = table['arm'] if table['method'].nunique()==1 else table['method'].map(display_names)
    fig, axes = plt.subplots(2,3,figsize=(15,8), constrained_layout=True)
    for ax, metric, caption in zip(axes.flat,
        ['answer_em','complete_evidence_hit','online_seconds','retrieval_seconds','llm_input_tokens','online_api_cost_usd'],
        ['정답 일치율 · 원래 EM','정답 근거 ID 집합 확보율','실제 전체 응답 시간 · 초/질문',
         '실제 검색 시간 · 초/질문','공통 생성기 API 입력 · 토큰/질문','온라인 API 비용 · USD/질문']):
        values = table[metric]
        method_colors=dict(zip(['A','B','C','D'],colors))
        bars = ax.bar(table.label, values, color=[method_colors[m] for m in table.method])
        ax.set_title(caption,loc='left',fontweight='bold'); ax.grid(axis='y',alpha=.18); ax.set_axisbelow(True)
        ax.spines[['top','right']].set_visible(False)
        if metric in ('answer_em','complete_evidence_hit'):
            ax.set_ylim(0,1.1); ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
            labels = [f'{v:.1%}' for v in values]
        elif metric.endswith('usd'):
            labels = [f'${v:.6f}' for v in values]
        else:
            labels = [f'{v:.2f}' for v in values]
        ax.bar_label(bars,labels=labels,padding=4)
        ax.tick_params(axis='x',labelsize=9)
    fig.suptitle(title+'\n동일 코퍼스·질문·생성기, 실제 실행 1회. 근거 ID 일치는 의미적 충분성과 다를 수 있음.',fontweight='bold')
    fig.savefig(destination,dpi=160)
    fig.savefig(Path(destination).with_suffix('.svg'))
    plt.show()
if dev_summary is not None:
    comparison_plot(dev_summary, 'C 개별효과 · 개발 40문항' if dev_summary.questions.iloc[0]==40 else 'C 개별효과 · 개발셋', EXPORT_DIR/'dev_c_comparison.png')
for frame in comparison_frames:
    split = frame['split'].iloc[0]
    arm = frame['arm'].iloc[0]
    names={'ABD_input_fix':'A/B/D · 입력 수정 대조','D_candidate':'D 후보 정책 · 개발','ABCD_frozen':'A/B/C/D · 별도 평가'}
    count=frame.question_id.nunique()
    comparison_plot(aggregate(frame), f'{names[arm]} {count}문항', EXPORT_DIR/f'{split}_{arm}_comparison.png')

## 5. 최종 평가의 질문별 분포와 대응 비교

시간 분포와 데이터셋별 답변 점수를 함께 본다. 대응 비교는 같은 질문의 C와 A/B 결과를 뺀다.
95% 구간은 NQ/HotpotQA의 질문을 각각 복원추출한 10,000회 부트스트랩이다.
질문 간 변동에 대한 보조 분석이며 API 시간대 변동이나 기업 자료 전체의 불확실성을 나타내지 않는다.
평가 결과를 보고 설정을 다시 바꾸지 않는다.

In [ ]:
evaluation = next((f for f in comparison_frames if f['split'].iloc[0]=='eval'), None)
if evaluation is not None:
    display(evaluation.groupby(['task','method']).agg(questions=('question_id','size'), EM=('answer_em','mean'),
        F1=('answer_f1','mean'), complete_gold_IDs=('complete_evidence_hit','mean'),
        seconds_mean=('online_seconds','mean'), seconds_p50=('online_seconds','median'),
        seconds_p95=('online_seconds',lambda values: values.quantile(.95))))
    fig, axes = plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
    methods = ['A','B','C','D']
    distributions = [evaluation.loc[evaluation.method==m,'online_seconds'] for m in methods]
    axes[0].boxplot(distributions,tick_labels=methods)
    axes[0].set_title('최종 평가 · 전체 시간 분포'); axes[0].set_ylabel('초 / 질문')
    tasks = evaluation.groupby(['task','method']).answer_em.mean().unstack()
    tasks[methods].plot.bar(ax=axes[1],color=colors,rot=0)
    axes[1].set_title('데이터셋별 정답 일치율'); axes[1].set_xlabel('')
    axes[1].set_ylim(0,1); axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1))
    for ax in axes: ax.grid(axis='y',alpha=.18); ax.set_axisbelow(True); ax.spines[['top','right']].set_visible(False)
    fig.savefig(EXPORT_DIR/'eval_distribution.png',dpi=160); plt.show()
    paired = []
    for metric in ['answer_em','complete_evidence_hit','online_seconds','online_api_cost_usd']:
        wide = evaluation.pivot(index=['task','question_id'],columns='method',values=metric)
        assert not wide.isna().any().any()
        for comparator in ['A','B']:
            deltas = wide['C']-wide[comparator]
            rng = np.random.default_rng(20261004)
            sample = [rng.choice(group.to_numpy(),size=(10000,len(group)),replace=True) for _,group in deltas.groupby(level='task')]
            bootstrap = np.concatenate(sample,axis=1).mean(axis=1)
            low,high = np.quantile(bootstrap,[.025,.975])
            paired.append({'comparison':f'C - {comparator}','metric':metric,'mean_difference':deltas.mean(),
                           'bootstrap_low':low,'bootstrap_high':high,'questions':len(deltas)})
    pd.DataFrame(paired).to_csv(EXPORT_DIR/'eval_paired_comparison.csv',index=False)
    display(pd.DataFrame(paired))

## 6. 실제 호출·비용과 입력 진단

API 비용은 물리 HTTP 요청별 ledger에서 다시 합산하고 각 질문의 비용과 대조한다. 복사된 응답이나 로컬 D 호출을 API 요청으로 세지 않는다.
D는 모든 실제 입력의 지시문·선택지·본문 잘림과 CPU 폴백을 확인한다. 단계 처리량은 기록된 토큰 수/단계 시간이며 다른 토크나이저의 토큰을 합산하지 않는다.
질문 실패의 분류는 근거 ID와 EM에 따른 관측 단계다. 근거 ID가 맞아도 내용이 답에 충분하다는 뜻은 아니다.

In [ ]:
if evaluation is not None:
    directory=STUDY_DIR/'eval/ABCD_frozen'
    evaluation_manifest=read_json(directory/'manifest.json')
    records=read_jsonl(directory/'system.jsonl')
    ledger=read_jsonl(ROOT/evaluation_manifest['raw_log']/'api_calls.jsonl')
    assert len({c['call_id'] for c in ledger})==len(ledger)
    ledger_frame=pd.DataFrame(ledger)
    online=ledger_frame[ledger_frame.phase=='system']
    assert online.api_cost_usd.notna().all(), 'Unknown costs must remain explicit.'
    charges=online.groupby('scope_id').api_cost_usd.sum().to_dict()
    throughput=[]; packing=[]; failures=[]
    for r in records:
        retrieval=r['retrieval']; method=retrieval['method']
        assert abs(charges[r['raw_log']['scope_id']]-r['online_api_cost_usd'])<1e-10
        if method in ('A','B'):
            throughput.append({'model':'RetroMAE query','tokens':retrieval['model_tokens']['retriever_query']['nonpad'],
                               'seconds':retrieval['seconds']['query_encode']})
        if method=='B':
            throughput.append({'model':'DeBERTa pairs','tokens':retrieval['model_tokens']['reranker_pairs']['nonpad'],
                               'seconds':retrieval['seconds']['rerank']})
        if method=='D':
            for call in retrieval['trace']['calls']:
                assert call['status']=='completed' and call.get('cpu_fallbacks',0)==0
                throughput.append({'model':'Laya inference','tokens':call['response']['usage']['input_tokens'],
                                   'seconds':call['inference_seconds']})
                for row in call['effective_inputs']:
                    encoded=row['encoding']
                    assert encoded['instruction_tokens_dropped']==0 and not any(encoded['option_tokens_dropped'])
                    assert not encoded['truncated'] and encoded['state_tokens_dropped']==0
                    packing.append({'tokens':encoded['input_tokens'],'dropped':encoded['state_tokens_dropped']})
        if r['answer_metrics']['answer_em']==1: category='EM 일치'
        elif retrieval['metrics']['complete_candidate_hit']==0: category='후보 근거 ID 미확보'
        elif retrieval['metrics']['complete_evidence_hit']==0: category='최종 선택 근거 ID 누락'
        else: category='근거 ID 확보 후 EM 불일치'
        failures.append({'method':method,'question_id':retrieval['question_id'],'category':category})
    cost_roles=online.groupby(['method','role']).agg(requests=('call_id','size'),USD=('api_cost_usd','sum'),HTTP_seconds=('http_seconds','sum'))
    cost_roles.to_csv(EXPORT_DIR/'eval_api_roles.csv'); display(cost_roles)
    through=pd.DataFrame(throughput).groupby('model').agg(tokens=('tokens','sum'),seconds=('seconds','sum'))
    through['recorded_tokens_per_second']=through.tokens/through.seconds
    through.to_csv(EXPORT_DIR/'eval_local_throughput.csv'); display(through)
    print('D input rows checked:',len(packing),'| maximum tokens:',max(p['tokens'] for p in packing),'| dropped tokens: 0 | CPU fallback: 0')
    failure_frame=pd.DataFrame(failures)
    failure_frame.to_csv(EXPORT_DIR/'eval_pipeline_observations.csv',index=False)
    display(pd.crosstab(failure_frame.method,failure_frame.category))
    returned={r['generation']['returned_model'] for r in records}
    print('Recorded returned generator IDs:',returned)
    print('Physical API ledger and question totals reconciled; local D calls remain separate.')

In [ ]:
if STUDY_DIR is not None:
    paths = [p for p in STUDY_DIR.glob('*.json') if p.name in {
        'study_snapshot.json','frozen_c_policy.json','frozen_d_policy.json',
        'd_input_replay_summary.json','decision_validation_summary.json','score_order_summary.json'}]
    paths += [p for p in STUDY_DIR.glob('*.csv') if p.name in {
        'decision_validation.csv','score_order_validation.csv','dev_c_summary.csv','dev_d_summary.csv'}]
    for frame in comparison_frames:
        directory = STUDY_DIR/frame['split'].iloc[0]/frame['arm'].iloc[0]
        paths += [directory/name for name in ('manifest.json','completed.json','system.jsonl','questions.csv','summary.csv')]
        log_path = ROOT/read_json(directory/'manifest.json')['raw_log']/'api_calls.jsonl'
        if log_path.exists(): paths.append(log_path)
    if dev_summary is not None:
        for arm in STUDY['variants']:
            directory = STUDY_DIR/'dev'/arm
            paths += [directory/name for name in ('manifest.json','completed.json','system.jsonl','questions.csv')]
    for path in paths:
        source_hashes[str(path.relative_to(ROOT))] = file_sha256(path)
    print('단계 검증 원본과 API ledger 해시:', len(source_hashes))
notebook_path = ROOT / "notebook/06_compare_results.ipynb"
source_notebook = nbformat.read(notebook_path, as_version=4)
source_digest = hashlib.sha256(json.dumps(
    [(cell.cell_type, cell.source) for cell in source_notebook.cells],
    ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()
comparison_manifest = {
    "mode": "read_only_comparison", "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_run": SYSTEM_RUN, "methods": METHODS, "source_hashes": source_hashes,
    "analysis_notebook": "notebook/06_compare_results.ipynb", "analysis_source_sha256": source_digest,
    "comparison_loader_sha256": file_sha256(ROOT / "rag_baselines/comparison.py"),
    "population": {"split": manifest["split"], "questions": question_count,
                   "repeats": repeats, "corpus_count": corpus_count, "records": len(df)},
    "aggregation": "arithmetic mean of balanced question/repeat observations; *_n = measured observations",
    "time_scope": "fresh retrieval through GPT response; same system run; A/B local, C remote Jev, D local Laya",
    "cost_scope": "external Jev retrieval + GPT generation API usage; excludes local GPU, shared preparation and human work",
    "token_scope": "provider/model-specific input tokens are kept separate; no cross-tokenizer sum",
    "missing_values": "unknown measurement or inapplicable stage; not zero",
    "generation_prices": manifest["generation"]["prices"], "jev_prices": manifest["c"]["config"]["prices"],
    "laya_model": manifest.get("d", {}).get("runtime"),
    "software": {"python": sys.version.split()[0], "pandas": pd.__version__, "matplotlib": matplotlib.__version__},
    "outputs": {p.name: file_sha256(p) for p in sorted(EXPORT_DIR.iterdir()) if p.is_file()},
}
write_json(EXPORT_DIR / "manifest.json", comparison_manifest)
print(f"그림·CSV·manifest 저장 위치: {EXPORT_DIR}")
display(Markdown("이 노트북은 API를 호출하지 않았다. 저장된 원시 결과를 검증해 비교 자료만 내보냈다."))

## 출처와 해석 범위

원본 결과·실행 설정·개발에서 고정한 정책과 원시 호출 로그를 대조했다. 06은 모델 추론과 API 호출을 수행하지 않는다.
그림과 CSV의 출처 해시는 새 분석 폴더의 `manifest.json`에 기록한다. 원시 자료는 각 실행의 `manifest.json` → `raw_log`에서 찾는다.

EM/F1·근거 ID 확보·입력 토큰·전체 시간·API 비용은 별개다. 근거 ID 일치는 의미적 충분성을 보장하지 않는다.
서로 다른 토크나이저의 토큰은 합산하지 않으며 API 비용은 GPU·문서 준비·사람 비용을 제외한다.
개발 결과는 정책 선택용이고 평가 결과는 고정한 정책의 관측이다. 반복 1회의 대응 구간은 질문 간 변동에 한정한다.
과거 개발 C와 A/B/D 대조군의 시간은 일부 겹쳤으며, 새 05의 단계 검증은 순차 실행한다.
축소 코퍼스·선택 질문의 결과를 전체 Wikipedia나 기업 전체 자료 성능으로 일반화하지 않는다.